# Healthcare Cost Analysis

This notebook analyzes a healthcare insurance dataset to identify the variables that most strongly influence individual medical costs.

**Workflow**
- Load and inspect the dataset
- Prepare variables for modeling
- Perform exploratory visualizations in Python
- Build a linear regression model
- Evaluate the model with **R²** and **RMSE**
- Review regression coefficients
- Inspect residuals
- Build a decision tree regressor
- Review decision tree feature importance
- Summarize key findings

**Dataset expected:** `insurance.csv` in the same folder as this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Optional styling
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

## 1. Load the Dataset

In [ ]:
df = pd.read_csv("insurance.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

## 2. Data Preparation

The original dataset includes categorical variables.  
For modeling, these are encoded into numeric values.

- `smoker`: yes = 1, no = 0
- `sex`: male = 1, female = 0

The `region` column is left out of the main regression model here to keep the model simple and interpretable. If you want, you can extend the notebook later with one-hot encoding for region.

In [ ]:
df_model = df.copy()

df_model["smoker"] = df_model["smoker"].map({"yes": 1, "no": 0})
df_model["sex"] = df_model["sex"].map({"male": 1, "female": 0})

df_model.head()

## 3. Exploratory Data Analysis

These visualizations help show the major patterns in the data before modeling.

In [ ]:
# Correlation heatmap without seaborn
corr_cols = ["age", "bmi", "children", "smoker", "sex", "charges"]
corr = df_model[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm")

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)

ax.set_title("Correlation Heatmap")
fig.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# Average healthcare cost by smoking status
avg_by_smoker = df.groupby("smoker")["charges"].mean().reindex(["no", "yes"])

plt.figure()
plt.bar(avg_by_smoker.index, avg_by_smoker.values)
plt.title("Average Healthcare Cost by Smoking Status")
plt.xlabel("Smoker")
plt.ylabel("Average Charges")
plt.tight_layout()
plt.show()

In [ ]:
# BMI vs healthcare cost
colors = df["smoker"].map({"yes": "tab:orange", "no": "tab:blue"})

plt.figure()
plt.scatter(df["bmi"], df["charges"], alpha=0.6, c=colors)
plt.title("BMI vs Healthcare Cost")
plt.xlabel("BMI")
plt.ylabel("Charges")
plt.tight_layout()
plt.show()

In [ ]:
# Age vs healthcare cost
plt.figure()
plt.scatter(df["age"], df["charges"], alpha=0.6, c=colors)
plt.title("Age vs Healthcare Cost")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of healthcare costs
plt.figure()
plt.hist(df["charges"], bins=30)
plt.title("Distribution of Healthcare Costs")
plt.xlabel("Charges")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Linear Regression Model

In [ ]:
X = df_model[["age", "bmi", "children", "smoker", "sex"]]
y = df_model["charges"]

lin_reg = LinearRegression()
lin_reg.fit(X, y)

predictions = lin_reg.predict(X)

coefficients = pd.Series(lin_reg.coef_, index=X.columns)
coefficients

## 5. Regression Evaluation Metrics

### R²
R² shows how much of the variation in healthcare charges is explained by the model.

### RMSE
RMSE shows the typical prediction error in the same units as the target variable, which here is dollars.

In [ ]:
r2 = r2_score(y, predictions)
rmse = np.sqrt(mean_squared_error(y, predictions))

print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:,.2f}")

In [ ]:
metrics_df = pd.DataFrame({
    "Metric": ["R²", "RMSE"],
    "Value": [r2, rmse]
})
metrics_df

## 6. Regression Coefficients

This chart shows the relative effect of each predictor in the linear regression model.

In [ ]:
coefficients.sort_values().plot(kind="barh")
plt.title("Regression Coefficients")
plt.xlabel("Coefficient Value")
plt.tight_layout()
plt.show()

## 7. Residual Analysis

The residual plot helps evaluate model fit. A good residual pattern is generally centered around zero with no strong systematic shape.

In [ ]:
residuals = y - predictions

plt.figure()
plt.scatter(predictions, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.title("Residual Plot")
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals")
plt.tight_layout()
plt.show()

## 8. Decision Tree Regressor

A decision tree can capture nonlinear relationships that linear regression may miss.

In [ ]:
tree_model = DecisionTreeRegressor(random_state=42, max_depth=4)
tree_model.fit(X, y)

tree_predictions = tree_model.predict(X)
tree_r2 = r2_score(y, tree_predictions)
tree_rmse = np.sqrt(mean_squared_error(y, tree_predictions))

print(f"Decision Tree R²: {tree_r2:.4f}")
print(f"Decision Tree RMSE: {tree_rmse:,.2f}")

## 9. Decision Tree Feature Importance

Feature importance shows which variables contribute most to the tree's predictions.

In [ ]:
feature_importance = pd.Series(tree_model.feature_importances_, index=X.columns).sort_values()

feature_importance.plot(kind="barh")
plt.title("Decision Tree Feature Importance")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

## 10. Key Findings

- **Smoking status** is the strongest driver of healthcare costs.
- **Age** and **BMI** also contribute meaningfully to higher charges.
- The linear regression model provides an interpretable baseline.
- The decision tree helps capture nonlinear structure and confirms the importance of the same core variables.
- **R²** and **RMSE** provide a clear summary of model fit and prediction error.

## 11. Optional Next Steps

To make this project even stronger later, you could:
- one-hot encode `region`
- create train/test splits
- compare out-of-sample performance
- tune the decision tree
- compare against random forest or gradient boosting